# ⚙️ FORGE — Engineering Intelligence System
### Agentic AI Framework · Itanta AI Hackathon

**From natural-language prompt → fully-verified, Dockerized software**

---
| Step | What happens | Time |
|------|-------------|------|
| 1 | Install dependencies | ~3 min |
| 2 | Load API keys from Colab Secrets | ~10 sec |
| 3 | Upload & extract the Forge project zip | ~30 sec |
| 4 | Launch dashboard OR run CLI | instant |
| 5 | Inspect results | — |

## Step 1 — Install Dependencies

In [ ]:
import subprocess, sys

packages = [
    'groq>=0.9.0', 'langgraph>=0.2.0', 'langchain-core>=0.3.0',
    'langsmith>=0.1.0', 'wandb>=0.17.0', 'ruff>=0.4.0',
    'pylint>=3.2.0', 'bandit>=1.7.0', 'pytest>=8.0.0',
    'pytest-asyncio>=0.23.0', 'pytest-json-report>=1.5.0',
    'gradio>=4.40.0', 'pyyaml>=6.0', 'httpx>=0.27.0',
    'tenacity>=8.3.0', 'typing-extensions>=4.12.0',
]

print('Installing Forge dependencies...')
result = subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '-q', '--upgrade'] + packages,
    capture_output=True, text=True
)
if result.returncode != 0:
    print('STDERR:', result.stderr[-2000:])
else:
    print('✅ All dependencies installed')

## Step 2 — Load API Keys from Colab Secrets

Add your keys via **left sidebar → 🔑 Secrets** before running this cell.

| Secret name | Required | Where to get |
|-------------|----------|--------------|
| `GROQ_API_KEY` | ✅ Yes | [console.groq.com](https://console.groq.com) — free |
| `WANDB_API_KEY` | Optional | [wandb.ai/authorize](https://wandb.ai/authorize) — free |
| `LANGSMITH_API_KEY` | Optional | [smith.langchain.com](https://smith.langchain.com) — free |

> Keys are read from Colab Secrets — they never appear in notebook output or history.

In [ ]:
import os
from google.colab import userdata

def _load_secret(name, required=False):
    try:
        val = userdata.get(name)
        if val and val.strip():
            os.environ[name] = val.strip()
            masked = val[:8] + '...' + val[-4:] if len(val) > 12 else '***'
            print(f'  ✅ {name} loaded ({masked})')
            return val.strip()
        raise ValueError('empty')
    except Exception as e:
        if required:
            raise RuntimeError(
                f'{name} is required but not found in Colab Secrets.\n'
                'Go to: left sidebar → 🔑 Secrets → Add new secret'
            )
        print(f'  ⏭️  {name} not set (optional — skipping)')
        return None

print('Loading API keys from Colab Secrets...')
_load_secret('GROQ_API_KEY',     required=True)
_load_secret('WANDB_API_KEY',    required=False)
_load_secret('LANGSMITH_API_KEY',required=False)

if os.environ.get('LANGSMITH_API_KEY'):
    os.environ['LANGCHAIN_API_KEY'] = os.environ['LANGSMITH_API_KEY']

# Quick connectivity check
print('\nVerifying Groq connection...')
try:
    from groq import Groq
    _r = Groq(api_key=os.environ['GROQ_API_KEY']).chat.completions.create(
        model='llama-3.1-8b-instant',
        messages=[{'role': 'user', 'content': 'Reply: OK'}],
        max_tokens=5,
    )
    print(f'✅ Groq connected — response: {_r.choices[0].message.content}')
except Exception as e:
    print(f'❌ Groq connection failed: {e}')
    print('   Check GROQ_API_KEY in Colab Secrets and re-run this cell.')

## Step 3 — Upload & Extract Forge Project

Run this cell — a file picker appears. Select **`forge_project_v2.zip`** from your machine.
Everything is extracted and set up automatically.

In [ ]:
import os, sys, zipfile, shutil
from pathlib import Path
from google.colab import files

FORGE_DIR  = '/content/forge'
EXTRACT_TO = '/content'

# ── Upload ────────────────────────────────────────────────────────────────────
print('📂 Select forge_project_v2.zip from your machine...')
uploaded = files.upload()

if not uploaded:
    raise RuntimeError('No file uploaded. Re-run this cell and select the zip.')

zip_name = list(uploaded.keys())[0]
zip_path = Path('/content') / zip_name
print(f'✅ Uploaded: {zip_name} ({zip_path.stat().st_size / 1024:.1f} KB)')

# ── Extract ───────────────────────────────────────────────────────────────────
if Path(FORGE_DIR).exists():
    shutil.rmtree(FORGE_DIR)
    print('🗑️  Cleared previous forge/ directory')

with zipfile.ZipFile(zip_path, 'r') as zf:
    zf.extractall(EXTRACT_TO)
print(f'✅ Extracted to {EXTRACT_TO}')

# ── Verify ────────────────────────────────────────────────────────────────────
required = ['main.py', 'config.yaml', 'agents', 'core', 'dashboard', 'patterns']
missing  = [r for r in required if not (Path(FORGE_DIR) / r).exists()]
if missing:
    raise RuntimeError(
        f'Extraction incomplete — missing: {missing}\n'
        "The zip must contain a top-level 'forge/' folder."
    )

# ── Setup ─────────────────────────────────────────────────────────────────────
if FORGE_DIR not in sys.path:
    sys.path.insert(0, FORGE_DIR)
os.chdir(FORGE_DIR)
Path('./outputs').mkdir(exist_ok=True)
Path('./generated_project').mkdir(exist_ok=True)

print(f'\n✅ Forge ready  —  cwd: {os.getcwd()}')
print('Contents:')
for item in sorted(Path(FORGE_DIR).iterdir()):
    print(f"  {'📁' if item.is_dir() else '📄'} {item.name}")

## Step 4A — Launch Dashboard (Recommended)

A **public share URL** will print below — open it in any browser.  
If the pipeline crashes, the dashboard shows a **red error banner** explaining exactly what went wrong and what to try.

In [ ]:
import sys, os
sys.path.insert(0, os.getcwd())

from core.llm import reset_client
reset_client()  # pick up GROQ_API_KEY from environment

from dashboard.app import launch
launch(share=True, port=7860)

## Step 4B — CLI / Batch Mode

In [ ]:
import sys, os
sys.path.insert(0, os.getcwd())
from core.llm import reset_client
reset_client()

PROMPT = """
Build a REST API for a personal expense tracker.
Users can create, read, update, and delete expenses.
Each expense has: amount, category, description, date, and tags.
Support filtering by date range and category.
Include monthly budget limits per category with strict validation.
"""

PRIORITY_WEIGHTS = {
    'speed': 0.15, 'quality': 0.30,
    'test_coverage': 0.30, 'security': 0.20, 'simplicity': 0.05,
}

from main import run_cli
final_state = run_cli(PROMPT.strip(), PRIORITY_WEIGHTS)

## Step 5 — Inspect Results

In [ ]:
import json
from pathlib import Path

s_path = Path('./outputs/summary.json')
if s_path.exists():
    s = json.loads(s_path.read_text())
    print('FORGE WORKFLOW SUMMARY\n' + '='*45)
    print(f"Tasks:     {s['tasks_completed']}/{s['tasks_total']} done  ({s['tasks_failed']} failed)")
    print(f"Files:     {s['files_generated']} generated")
    print(f"Tests:     {s['tests_passed']}/{s['tests_total']} passed  ({s['test_pass_rate']}%)")
    print(f"Security:  {s['security_findings_total']} findings  ({s['security_findings_fixed']} fixed)")
    print(f"API calls: {s['api_calls_total']}  |  Tokens: {s['total_tokens']:,}")
    print(f"Tier:      {s['complexity_tier']}")
else:
    print('No summary yet — run Step 4B first.')

In [ ]:
from pathlib import Path

project_dir = Path('./generated_project')
all_files = sorted(f for f in project_dir.rglob('*') if f.is_file())
print(f'{len(all_files)} files in ./generated_project/\n')
for f in all_files:
    rel   = f.relative_to(project_dir)
    lines = len(f.read_text(errors='ignore').splitlines()) if f.suffix == '.py' else 0
    print(f"  {str(rel):<50} {f.stat().st_size:>6} bytes" + (f"  {lines} lines" if lines else ''))

In [ ]:
from pathlib import Path

FILE_TO_VIEW = 'app/main.py'   # change this
path = Path('./generated_project') / FILE_TO_VIEW
if path.exists():
    print(f'{'='*55}\n  {FILE_TO_VIEW}\n{'='*55}')
    print(path.read_text())
else:
    print(f'Not found: {FILE_TO_VIEW}\nAvailable:')
    for f in sorted(Path('./generated_project').rglob('*.py')):
        print(f'  {f.relative_to("./generated_project")}')

In [ ]:
# Decision audit trail
if 'final_state' in dir() and final_state:
    for d in final_state.get('decision_audit', []):
        c = d.get('confidence', 0)
        print(f"\n{d.get('agent')}  [{('█'*(c//10) + '░'*(10-c//10))}] {c}%")
        print(f"  {d.get('reasoning','')[:180]}")
        for a in d.get('alternatives', []):
            print(f'  • {a}')
else:
    print('Run Step 4B first.')

## Step 6 — All 5 Complexity Tiers

In [ ]:
import json, shutil
from pathlib import Path
from main import run_cli
from core.llm import reset_client

TIER_PROMPTS = {
    1: 'Build a REST API for a personal book library. Users can add, update, delete, search books. Fields: title, author, ISBN, genre, read_status, rating (1-5). Strict schema validation.',
    2: 'Build a dynamic pricing engine. Apply configurable rules: regional tax rates, volume discounts, seasonal multipliers, loyalty pricing. Rules loaded from JSON. API to evaluate prices and audit applied rules.',
    3: 'Build an async weather aggregation service. Fetch from two providers simultaneously with httpx. Implement: exponential backoff, TTL caching, graceful degradation. Return merged normalised data.',
    4: 'Build OAuth2 + JWT authentication with RBAC. Roles: admin, editor, viewer. bcrypt passwords. RS256 JWT, 15-min expiry. Registration, login, refresh, logout. Admin role management.',
    5: 'Build a MongoDB join engine: INNER, LEFT, RIGHT, FULL OUTER JOIN via aggregation pipelines. Sync historical data, then live Change Streams. REST API for join queries with field projection.',
}
TIER_NAMES = {1:'The Ledger',2:'Logic Engine',3:'Live Bridge',4:'The Gatekeeper',5:'Mongo-SQL Engine'}

TIER = 1  # change to 1-5

print(f'\n🚀 Forge Tier {TIER}: {TIER_NAMES[TIER]}\n')
reset_client()
shutil.rmtree('./generated_project', ignore_errors=True)
Path('./generated_project').mkdir(exist_ok=True)

state = run_cli(TIER_PROMPTS[TIER], {'speed':0.15,'quality':0.30,'test_coverage':0.30,'security':0.20,'simplicity':0.05})

if state and state.get('workflow_summary'):
    out = Path(f'./outputs/tier_{TIER}_summary.json')
    out.write_text(json.dumps(state['workflow_summary'], indent=2))
    print(f'\n✅ Summary: {out}')

## Step 7 — Download Generated Project

In [ ]:
import shutil
from pathlib import Path
from google.colab import files

shutil.make_archive('/content/forge_output', 'zip', '/content', 'generated_project')
print(f"Archive: {Path('/content/forge_output.zip').stat().st_size/1024:.1f} KB")
files.download('/content/forge_output.zip')

---
## Quick Reference

### Dashboard crash messages
| Icon | Cause | Fix |
|------|-------|-----|
| 🚦 | Rate limit | Wait 60s, retry |
| 📏 | Token limit exceeded | Simplify the prompt |
| 🔑 | API key missing/invalid | Check Colab Secrets |
| 🌐 | Network / timeout | Check internet |
| 🧩 | LLM returned bad JSON | Re-run (usually self-corrects) |
| ⚠️ | Unexpected error | Check Activity Log |

### Confidence routing
| Score | Behaviour |
|-------|-----------|
| ≥ 80% | Proceed autonomously |
| 60–79% | Proceed, flagged in dashboard |
| < 60% | Pause, request human input |

### Models
| Role | Model |
|------|-------|
| Planning + codegen | `llama-3.3-70b-versatile` |
| Boilerplate / fast calls | `llama-3.1-8b-instant` |